<a href="https://colab.research.google.com/github/Andres931720/Deberes/blob/main/Seminario_Algoritmos_Andr%C3%A9s_Vinueza_Machuca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Seminario<br>
Nombre y Apellidos: Andrés Vinueza Machuca  <br>
Url: <br>https://drive.google.com/file/d/1Qp1eRxeVBB-Jx2OaO0XHcJIS5J1S02sh/view?usp=sharing


Problema:
> 1. Sesiones de doblaje <br>
>2. Organizar los horarios de partidos de La Liga<br>
>3. Combinar cifras y operaciones

Descripción del problema:

Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de grabación independientemente del número de tomas que se graben. No es posible grabar más de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor posible. Los datos son:

Número de actores: 10

Número de tomas : 30

Actores/Tomas : https://bit.ly/36D8IuK

1 indica que el actor participa en la toma

0 en caso contrario




                                        

In [ ]:
#Librerías
import pandas as pd
import numpy as np
#Leemos la base]
datos = pd.read_excel("Datos problema doblaje(30 tomas, 10 actores).xlsx",
                      skiprows=1,index_col=0)

In [ ]:
datos = datos.iloc[0:30, 0:10]

In [ ]:
print(datos)

        1    2    3    4    5    6    7    8    9   10
Toma                                                  
1     1.0  1.0  1.0  1.0  1.0  0.0  0.0  0.0  0.0  0.0
2     0.0  0.0  1.0  1.0  1.0  0.0  0.0  0.0  0.0  0.0
3     0.0  1.0  0.0  0.0  1.0  0.0  1.0  0.0  0.0  0.0
4     1.0  1.0  0.0  0.0  0.0  0.0  1.0  1.0  0.0  0.0
5     0.0  1.0  0.0  1.0  0.0  0.0  0.0  1.0  0.0  0.0
6     1.0  1.0  0.0  1.0  1.0  0.0  0.0  0.0  0.0  0.0
7     1.0  1.0  0.0  1.0  1.0  0.0  0.0  0.0  0.0  0.0
8     1.0  1.0  0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0
9     1.0  1.0  0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0
10    1.0  1.0  0.0  0.0  0.0  1.0  0.0  0.0  1.0  0.0
11    1.0  1.0  1.0  0.0  1.0  0.0  0.0  1.0  0.0  0.0
12    1.0  1.0  1.0  1.0  0.0  1.0  0.0  0.0  0.0  0.0
13    1.0  0.0  0.0  1.0  1.0  0.0  0.0  0.0  0.0  0.0
14    1.0  0.0  1.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0
15    1.0  1.0  0.0  0.0  0.0  0.0  1.0  0.0  0.0  0.0
16    0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0  0.0  1.0
17    1.0 

### **(*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?**



Respuesta

Sin restricciones podemos considerar que por cada día un actor puede ir a grabar una toma. Si tenemos 30 tomas podemos permutar cada toma, entonces tenemos $30! $

### **¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones.**


Respuesta

Con restricciones podemos hacer una combinación sin repeticiones y sin un orden para 6 de 30 tomas en grupos de 5

$$ \frac{30!}{(6!)^5 5!} \approx 11423951396577720 $$

## Modelo para el espacio de soluciones<br>
## (*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, arguentalo)


Respuesta

Inicialmente escogí como estructura de datos una `list(tuple)`, donde cada elemento de la lista es una tupla con los actores de una toma. Al trabajar con ella encontré dos problemas. El primero es que la tupla impone un orden entre sus elementos, y ese orden no aporta nada al problema: dos tomas con los mismos actores en distinto orden son la misma toma, pero como estructuras son distintas, lo que puede llevar a tratar como diferentes cosas que en realidad son equivalentes y hacer que el problema crezca con operaciones innecesarias. El segundo es que calcular el costo de un día —que depende de qué actores participan— obliga a comprobar pertenencia dentro de las tuplas, una operación más costosa de lo necesario.

Por esto opté por una `list(set)`, donde cada elemento de la lista representa una toma y cada `set` es el conjunto de actores que participan en ella. Esta estructura resuelve los dos problemas anteriores.

La primera ventaja es que el `set` no tiene orden: un conjunto de actores es el mismo sin importar cómo se escriban sus elementos, así que desaparece la ambigüedad que introducía la tupla y no corro el riesgo de tratar como distintas dos tomas que son iguales.

La segunda ventaja es que los cálculos de costo se vuelven directos. El costo de un día es el número de actores distintos que asisten, es decir, el tamaño de la unión de los conjuntos de sus tomas; y el costo incremental de añadir una toma a un día es cuántos actores nuevos aporta, es decir, una diferencia de conjuntos. Ambas operaciones —unión, diferencia y pertenencia— son nativas y eficientes sobre `set`, mientras que sobre tuplas habría que implementarlas de forma más costosa.

Además, esta representación se complementa bien con el diseño del recorrido del algoritmo. Como cada toma se identifica por su conjunto de actores y no por una posición, junto con la forma en que ancla la construcción de cada día (la toma pivote de menor índice), el algoritmo no vuelve a visitar una misma agrupación por caminos distintos: cada solución se genera una sola vez. De este modo, la estructura no solo simplifica los cálculos, sino que también evita recorrer y almacenar en memoria ramas que corresponden a soluciones ya exploradas.

## Según el modelo para el espacio de soluciones<br>
## (*)¿Cual es la función objetivo?

## (*)¿Es un problema de maximización o minimización?

##
Conjuntos

$A$: conjunto de actores de doblaje

$T$: conjunto de tomas

$C$: conjunto de días disponibles (candidatos), $|C| = |T|$

#### Parámetros

$P$: costo por actor por día de desplazamiento al estudio

$n$: número máximo de tomas grabables por día ($n = 6$)

$d_{at}$: 1 si el actor $a$ participa en la toma $t$, 0 en otro caso

#### Variables de decisión

$x_{ac} \in \{0,1\}$: 1 si el actor $a$ debe asistir al estudio el día $c$, 0 en otro caso

$y_{tc} \in \{0,1\}$: 1 si la toma $t$ se graba el día $c$, 0 en otro caso

#### Función objetivo

Minimizar el costo total en desplazamientos de actores:

$$
\min \quad P \sum_{a \in A} \sum_{c \in C} x_{ac}
$$

#### Restricciones

**(1) Cada toma se graba exactamente un día:**

$$
\sum_{c \in C} y_{tc} = 1 \qquad \forall t \in T
$$

**(2) Capacidad máxima de tomas por día:**

$$
\sum_{t \in T} y_{tc} \leq n \qquad \forall c \in C
$$

**(3) Un actor asiste el día $c$ si alguna de sus tomas se graba ese día:**

$$
x_{ac} \geq d_{at} \, y_{tc} \qquad \forall a \in A,\ \forall t \in T,\ \forall c \in C
$$

**(4) Naturaleza binaria de las variables:**

$$
x_{ac} \in \{0,1\} \qquad \forall a \in A,\ \forall c \in C
$$

$$
y_{tc} \in \{0,1\} \qquad \forall t \in T,\ \forall c \in C
$$

## Diseña un algoritmo para resolver el problema por fuerza bruta

Respuesta

**Entradas:** conjunto de tomas $T = \{0, \dots, |T|-1\}$, capacidad diaria $n$, función $\text{calcular\_costo}(\text{día})$

**Salida:** horario de costo mínimo

**Pasos**

**(1) Inicialización**

$$
\text{tomas} = \{0, \dots, |T|-1\}, \quad \text{horario} = [\,], \quad \text{costo} = 0, \quad \text{mejor\_costo} = \infty
$$

**(2) Caso base**

Si $\text{tomas} = \varnothing$, comparamos la solución actual contra la mejor encontrada hasta el momento:

$$
\text{mejor\_costo} = \min(\text{mejor\_costo}, \ \text{costo})
$$

y actualizamos $\text{mejor\_horario}$ si $\text{costo} < \text{mejor\_costo}$ previo. Terminamos esta rama de la recursión.

**(3) Toma pivote**

$$
t_0 = \min(\text{tomas})
$$

**(4) Combinaciones factibles**

Generamos todos los subconjuntos de las tomas restantes (sin incluir $t_0$) de tamaño hasta $n-1$, para completar el día junto a $t_0$:

$$
\text{Combinaciones} = \left\{ X \subseteq (\text{tomas} \setminus \{t_0\}) \ : \ |X| \leq n-1 \right\}
$$

**(5) Exploración de cada combinación**

Para cada $X \in \text{Combinaciones}$:

$$
\text{día} = \{t_0\} \cup X
$$

$$
c = \text{calcular\_costo}(\text{día})
$$

$$
\text{tomas}' = \text{tomas} \setminus \text{día}, \qquad \text{costo}' = \text{costo} + c, \qquad \text{horario}' = \text{horario} \cup \{\text{día}\}
$$

**(6) Recursión**

Se repite el algoritmo (paso 2 en adelante) sobre $\text{tomas}'$, $\text{costo}'$ y $\text{horario}'$, para cada combinación generada en el paso 5.


---
*Resumen del algoritmo*

El algoritmo genera todas las posibles combinaciones de tomas a grabar en un día de tamaño $\leq n$, luego combina todos los conjuntos de días para generar el calendario completo de grabaciones. En cada iteración se evalúa el coste de cada una y se queda con la mínima. Al recorrer el espacio completo, la optimalidad está garantizada por construcción.

El detalle clave es la **toma pivote** del paso (3). Al forzar que el día "actual" contenga siempre $t_0 = \min(\text{tomas})$, cada día queda *anclado* por su toma de menor índice. Esto evita generar la misma partición varias veces en distinto orden: sin este truco, el día $\{0,1\}$ y el día $\{1,0\}$ se explorarían como ramas distintas, y la partición $\{\{0,1\},\{2,3\}\}$ aparecería mezclada con $\{\{2,3\},\{0,1\}\}$. Con el pivote, cada partición se genera **una sola vez**. De esta forma se reduce la redundancia, pero sigue explorando el árbol entero.

## Calcula la complejidad del algoritmo por fuerza bruta

Respuesta

Sea $T = |T|$ (número de tomas), $A = |A|$ (número de actores) y $n$ la capacidad diaria.

Sea $L(T,n)$ la función que cuenta las soluciones (particiones completas) exploradas por el algoritmo.

**Caso base:**
$$
L(0,n) = 1
$$

**Recursión general** (fijando en cada nivel la toma pivote $t_0$ y variando el tamaño $k$ del día que la contiene):
$$
L(T,n) = \sum_{k=1}^{\min(n,T)} \binom{T-1}{k-1}\, L(T-k,\,n)
$$

**Caso particular — solo bloques de tamaño exacto $n$** (subconjunto restringido del espacio de búsqueda, válido únicamente si el algoritmo se limita a explorar días completos, y solo cuando $n \mid T$):
$$
L(T,n) = \frac{T!}{(n!)^{T/n}\,(T/n)!}
$$

**Complejidad temporal**

El trabajo por cada solución completa (hoja) consiste en calcular el costo de cada día de la partición — unión de actores por toma — lo que en total, sumado sobre los $\approx T/n$ días de una partición, cuesta $O(T \cdot A)$. Por lo tanto:
$$
\text{Tiempo total} = O\big(L(T,n)\cdot T \cdot A\big)
$$

### Desarrollo del algoritmo de fuerza bruta

In [ ]:

def df_a_tomas_actores(df: pd.DataFrame, mantener_nombres: bool = False):
    """
    Convierte un DataFrame (filas = tomas, columnas = actores) en la estructura
    de trabajo de los algoritmos.

    Parámetros
    ----------
    df : pd.DataFrame
        Filas = tomas, columnas = actores. Cada celda indica si el actor
        participa en la toma. Admite valores booleanos, numéricos o de texto.
    mantener_nombres : bool
        Si es True, los conjuntos contienen los NOMBRES de los actores.
        Si es False (por defecto), contienen sus ÍNDICES enteros (0..A-1).

    Devuelve
    --------
    tomas_actores : list[set]
        Lista de conjuntos; cada conjunto es una toma con los actores que
        participan en ella. Este es el resultado principal.
    n_actores : int
        Número total de actores (A).
    actor_a_idx : dict
        Mapeo {nombre_actor -> índice}, útil para interpretar los resultados.
    """
    # 1) Normalizar cada columna a booleano (presente / ausente),
    #    tolerando mezcla de tipos.
    cols_bool = {}
    for col in df.columns:
        s = df[col]
        if pd.api.types.is_bool_dtype(s):
            cols_bool[col] = s.fillna(False)
        elif pd.api.types.is_numeric_dtype(s):
            cols_bool[col] = s.fillna(0).gt(0)                # > 0 => presente
        else:
            s_norm = s.fillna("").astype(str).str.strip().str.lower()
            ausentes = {"", "0", "false", "no", "nan", "none"}
            cols_bool[col] = ~s_norm.isin(ausentes)           # no vacío => presente

    df_bool = pd.DataFrame(cols_bool, index=df.index)

    # 2) Mapeo actor -> índice (0..A-1).
    actores = list(df_bool.columns)
    actor_a_idx = {actor: i for i, actor in enumerate(actores)}
    n_actores = len(actores)

    # 3) Construir la lista de conjuntos de forma vectorizada (sin iterrows).
    matriz = df_bool.to_numpy(dtype=bool)                     # (n_tomas x A)
    if mantener_nombres:
        actores_arr = np.array(actores, dtype=object)
        tomas_actores = [set(actores_arr[fila]) for fila in matriz]
    else:
        indices = np.arange(n_actores)
        tomas_actores = [set(indices[fila].tolist()) for fila in matriz]

    return tomas_actores, n_actores, actor_a_idx

In [ ]:
tomas_actores = df_a_tomas_actores(datos)

In [ ]:
tomas_actores

([{0, 1, 2, 3, 4},
  {2, 3, 4},
  {1, 4, 6},
  {0, 1, 6, 7},
  {1, 3, 7},
  {0, 1, 3, 4},
  {0, 1, 3, 4},
  {0, 1, 5},
  {0, 1, 3},
  {0, 1, 5, 8},
  {0, 1, 2, 4, 7},
  {0, 1, 2, 3, 5},
  {0, 3, 4},
  {0, 2, 5},
  {0, 1, 6},
  {3, 9},
  {0, 2},
  {2, 5},
  {0, 2},
  {0, 2, 3, 4},
  {5, 7},
  {0, 1, 2, 3},
  {0, 2},
  {2, 5},
  {0, 1, 3, 9},
  {0, 2, 4, 8},
  {3, 4},
  {0, 3},
  {0, 4, 5},
  {0, 3}],
 10,
 {1: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5, 7: 6, 8: 7, 9: 8, 10: 9})

Por el problema a repartir se incluye en la programación un límite inferior de días para restringir la exploración. No es necesario evaluar días de longitud 1.

In [ ]:
from itertools import combinations


def costo_dia(dia, tomas_actores):
    """Coste de un día = número de actores DISTINTOS que asisten ese día."""
    actores = set()
    for t in dia:
        actores |= tomas_actores[t]
    return len(actores)


def fuerza_bruta(tomas_actores, n, min_dia=1, verbose=True):
    """
    Resuelve el problema de tomas de doblaje por ENUMERACIÓN EXHAUSTIVA.

    Genera todas las particiones de las tomas en días de tamaño en el rango
    [min_dia, n], usando una "toma pivote" (la de menor índice pendiente)
    para evitar particiones repetidas, y devuelve la de coste mínimo.

    Parámetros
    ----------
    tomas_actores : list[set]
        Lista de conjuntos; cada conjunto es una toma con sus actores.
    n : int
        Capacidad diaria (máximo de tomas por día).
    min_dia : int
        Tamaño MÍNIMO de día. Con esto se restringe el espacio de búsqueda:
          - min_dia = 1  -> comportamiento estándar (permite días de 1 toma).
          - min_dia = 2  -> prohíbe días de una sola toma.
          - min_dia = n  -> solo días LLENOS (útil cuando |T| es múltiplo de n
                            y se asume que el óptimo tiene todos los días llenos).
        Nota: fijar min_dia > 1 es una decisión de modelado que restringe el
        espacio; en casos patológicos (actores muy disjuntos) podría dejar
        fuera un óptimo con días pequeños. Para la instancia de doblaje típica
        no ocurre, y acelera mucho la búsqueda.
    verbose : bool
        Si es True, imprime cada solución temporal que mejora a la anterior.

    Devuelve
    --------
    mejor_horario : list[list[int]]
    mejor_costo : int
    nodos : int
    """
    T = len(tomas_actores)
    mejor = {"costo": float("inf"), "horario": None}
    nodos = 0
    mejoras = 0

    def explorar(pendientes, horario, costo):
        nonlocal nodos, mejoras
        nodos += 1

        # (2) Caso base: partición completa.
        if not pendientes:
            if costo < mejor["costo"]:
                mejor["costo"] = costo
                mejor["horario"] = [list(d) for d in horario]
                mejoras += 1
                if verbose:
                    print(f"[mejora {mejoras}] nodo {nodos:>6} | "
                          f"costo = {costo} | horario = {mejor['horario']}")
            return

        # Poda: si quedan menos tomas que el mínimo, no se puede formar
        # ningún día válido -> rama muerta.
        if len(pendientes) < min_dia:
            return

        # (3) Toma pivote.
        t0 = min(pendientes)
        resto = sorted(pendientes - {t0})

        # (4) Combinaciones factibles. Como |dia| = 1 + |X| y se exige
        #     min_dia <= |dia| <= n, entonces min_dia-1 <= |X| <= n-1.
        k_min = max(0, min_dia - 1)
        k_max = min(n - 1, len(resto))
        for k in range(k_min, k_max + 1):
            for X in combinations(resto, k):
                dia = [t0] + list(X)                 # (5)
                c = costo_dia(dia, tomas_actores)
                explorar(pendientes - set(dia),      # (6) recursión
                         horario + [dia],
                         costo + c)

    # (1) Inicialización.
    explorar(set(range(T)), [], 0)

    if verbose:
        print(f"=> Óptimo: costo = {mejor['costo']} | nodos = {nodos} "
              f"| mejoras = {mejoras}")

    return mejor["horario"], mejor["costo"], nodos




In [ ]:
fuerza_bruta(tomas_actores[0], n=6, min_dia=6)

[mejora 1] nodo      6 | costo = 38 | horario = [[0, 1, 2, 3, 4, 5], [6, 7, 8, 9, 10, 11], [12, 13, 14, 15, 16, 17], [18, 19, 20, 21, 22, 23], [24, 25, 26, 27, 28, 29]]
[mejora 2] nodo      8 | costo = 37 | horario = [[0, 1, 2, 3, 4, 5], [6, 7, 8, 9, 10, 11], [12, 13, 14, 15, 16, 17], [18, 19, 20, 21, 22, 24], [23, 25, 26, 27, 28, 29]]
[mejora 3] nodo     70 | costo = 36 | horario = [[0, 1, 2, 3, 4, 5], [6, 7, 8, 9, 10, 11], [12, 13, 14, 15, 16, 17], [18, 19, 20, 22, 23, 28], [21, 24, 25, 26, 27, 29]]
[mejora 4] nodo    552 | costo = 35 | horario = [[0, 1, 2, 3, 4, 5], [6, 7, 8, 9, 10, 11], [12, 13, 14, 15, 16, 17], [18, 20, 22, 23, 25, 28], [19, 21, 24, 26, 27, 29]]
[mejora 5] nodo   6697 | costo = 34 | horario = [[0, 1, 2, 3, 4, 5], [6, 7, 8, 9, 10, 11], [12, 13, 14, 15, 16, 24], [17, 18, 20, 22, 23, 28], [19, 21, 25, 26, 27, 29]]
[mejora 6] nodo  52787 | costo = 33 | horario = [[0, 1, 2, 3, 4, 5], [6, 7, 8, 9, 10, 11], [12, 13, 14, 15, 21, 24], [16, 17, 18, 20, 22, 23], [19, 25, 26,

KeyboardInterrupt: 

Como se observa en el cálculo del primer apartado, el número de combinaciones posibles crece de forma explosiva, lo que hace inviable ejecutar la fuerza bruta sobre la instancia completa: aunque garantiza el óptimo, su costo computacional es impracticable. Esta misma limitación la reportan Brisaboa et al. (2015) para este problema, donde incluso un método exacto basado en programación lineal binaria requiere tiempos de resolución de hasta decenas de horas en instancias reales, motivo por el cual recurren a una heurística. Por ello, la evaluación exhaustiva se aplca para validar el óptimo en instancias reducidas.

In [ ]:
# --------------------------------------------------------------------
# Ejemplo de uso:
#
#   tomas_actores, A, actor_a_idx = df_a_tomas_actores(df)
#   # Para tu tarea (|T| múltiplo de n, días llenos):
#   horario, costo, nodos = fuerza_bruta(tomas_actores, n=6, min_dia=6)
# --------------------------------------------------------------------

tomas = [
    {0, 1, 2}, {0, 1}, {3, 4}, {2, 3}, {4, 5}, {0, 5},
]
print("--- min_dia=1 ---")
fuerza_bruta(tomas, n=3, min_dia=1)
print("\n--- min_dia=3 (solo días llenos) ---")
fuerza_bruta(tomas, n=3, min_dia=3)

--- min_dia=1 ---
[mejora 1] nodo      7 | costo = 13 | horario = [[0], [1], [2], [3], [4], [5]]
[mejora 2] nodo      8 | costo = 12 | horario = [[0], [1], [2], [3], [4, 5]]
[mejora 3] nodo     17 | costo = 11 | horario = [[0], [1], [2, 3], [4, 5]]
[mejora 4] nodo     71 | costo = 10 | horario = [[0], [1, 5], [2, 3, 4]]
[mejora 5] nodo    110 | costo = 9 | horario = [[0, 1], [2, 3], [4, 5]]
[mejora 6] nodo    260 | costo = 8 | horario = [[0, 1, 3], [2, 4, 5]]
=> Óptimo: costo = 8 | nodos = 340 | mejoras = 6

--- min_dia=3 (solo días llenos) ---
[mejora 1] nodo      3 | costo = 10 | horario = [[0, 1, 2], [3, 4, 5]]
[mejora 2] nodo      5 | costo = 8 | horario = [[0, 1, 3], [2, 4, 5]]
=> Óptimo: costo = 8 | nodos = 21 | mejoras = 2


([[0, 1, 3], [2, 4, 5]], 8, 21)

## (*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

Respuesta

### Algoritmo 1 — Voraz

**Entradas:** conjunto de tomas $T = \{0, \dots, |T|-1\}$, capacidad diaria $n$, función $\text{actores}(t)$ que devuelve el conjunto de actores de la toma $t$.

**Salida:** solución del conjunto de tomas por día con costo bajo, **no se garantiza de optimalidad**.

**(1) Inicialización**

$$
\text{pendientes} = \{0, \dots, |T|-1\}, \quad \text{horario} = [\,], \quad \text{costo} = 0
$$

**(2) Bucle principal.** Mientras $\text{pendientes} \neq \varnothing$, construimos un día completo:

**(2a) Semilla del día.** Elegimos la toma pendiente con **más actores**, para tener controlada desde el inicio la toma más "cara":

$$
\text{semilla} = \arg\max_{t \,\in\, \text{pendientes}} |\text{actores}(t)|
$$

$$
\text{día} = [\text{semilla}], \quad A_{\text{día}} = \text{actores}(\text{semilla}), \quad \text{pendientes} \leftarrow \text{pendientes} \setminus \{\text{semilla}\}
$$

**(2b) Relleno del día.** Mientras $|\text{día}| < n$ y queden pendientes, añadimos la toma que aporte **menos actores nuevos** a los que ya están ese día:

$$
t^{*} = \arg\min_{t \,\in\, \text{pendientes}} \big|\,\text{actores}(t) \setminus A_{\text{día}}\,\big|
$$

$$
\text{día} \leftarrow \text{día} \cup \{t^{*}\}, \quad A_{\text{día}} \leftarrow A_{\text{día}} \cup \text{actores}(t^{*}), \quad \text{pendientes} \leftarrow \text{pendientes} \setminus \{t^{*}\}
$$

**(2c) Cierre del día.** Añadimos el día al horario y acumulamos su coste (actores distintos que asisten ese día):

$$
\text{horario} \leftarrow \text{horario} \cup \{\text{día}\}, \quad \text{costo} \leftarrow \text{costo} + |A_{\text{día}}|
$$

**(3) Salida.** Devolver $\text{horario}$ y $\text{costo}$.

---
*Resumen del algoritmo*

La clave de la optimización es reducir el número de **actores distintos** que asisten, porque a cada actor se le paga una vez por día. Por eso, al rellenar un día, lo que de verdad cuesta dinero al meter una toma no es cuántos actores comparte con el día, sino cuántos actores **nuevos** trae.

Este matiz es importante: *máxima intersección* $\neq$ *mínimos actores nuevos*, porque las tomas tienen tamaños distintos. Si el día ya tiene $\{1,2,3,4\}$ y comparamos una toma $\{1,2,3,5,6\}$ (intersección 3, pero 2 nuevos) contra $\{1,4\}$ (intersección 2, pero 0 nuevos), conviene la segunda. Por eso el criterio de (2b) minimiza $|\text{actores}(t) \setminus A_{\text{día}}|$ y no la intersección.


El lagoritmo voraz propuesto toma la mejor decisión **local** en cada paso, pero eso **no garantiza el óptimo global**. Es la misma lección del problema del cambio de monedas: con denominaciones $\{1, 5, 11\}$ para 15 céntimos, el voraz da $11 + 1 + 1 + 1 + 1 = 5$ monedas, cuando el óptimo es $5 + 5 + 5 = 3$. Aquí ocurre lo análogo: la semilla o el relleno pueden llevar a una agrupación que parece buena localmente pero que impide una mejor combinación más adelante.

Usaremos el voraz como **solución rápida y cota superior inicial**, no como respuesta final. Su valor está en dar un buen punto de partida para el algoritmo exacto.

### Desarrollo del algoritmo  voraz

In [ ]:
def voraz(tomas_actores, n, verbose=True):
    """
    Resuelve (de forma aproximada) el problema de tomas de doblaje con una
    estrategia VORAZ.

    Construye los días uno a uno:
      - Semilla: cada día se inicia con la toma pendiente de MÁS actores
        (la más "cara", para tenerla controlada desde el inicio).
      - Relleno: mientras el día no llegue a n tomas, se añade la toma que
        aporte MENOS actores nuevos a los que ya están ese día.

    El criterio de relleno minimiza los actores NUEVOS (no la intersección),
    porque lo que cuesta dinero al meter una toma es cuántos actores nuevos
    trae, no cuántos comparte. Como el voraz toma la mejor decisión local en
    cada paso, NO garantiza el óptimo global; se usa como solución rápida y
    como cota superior inicial para el algoritmo de ramificación y poda.

    Parámetros
    ----------
    tomas_actores : list[set]
        Lista de conjuntos; cada conjunto es una toma con sus actores.
    n : int
        Capacidad diaria (máximo de tomas por día).
    verbose : bool
        Si es True, imprime la composición de cada día.

    Devuelve
    --------
    horario : list[list[int]]
        Lista de días; cada día es la lista de índices de tomas.
    costo : int
        Coste total (suma de actores distintos por día).
    """
    T = len(tomas_actores)
    pendientes = set(range(T))
    horario = []
    costo = 0

    while pendientes:
        # (2a) Semilla: toma pendiente con MÁS actores.
        semilla = max(pendientes, key=lambda t: len(tomas_actores[t]))
        dia = [semilla]
        actores_dia = set(tomas_actores[semilla])
        pendientes.discard(semilla)

        # (2b) Relleno: la toma que aporte MENOS actores nuevos.
        while len(dia) < n and pendientes:
            t_mejor = min(
                pendientes,
                key=lambda t: len(tomas_actores[t] - actores_dia)
            )
            dia.append(t_mejor)
            actores_dia |= tomas_actores[t_mejor]
            pendientes.discard(t_mejor)

        # (2c) Cierre del día: acumular su coste (actores distintos).
        horario.append(dia)
        costo += len(actores_dia)
        if verbose:
            print(f"  día {len(horario)}: tomas {dia} | "
                  f"actores {sorted(actores_dia)} | +{len(actores_dia)}")

    if verbose:
        print(f"=> Voraz: costo = {costo} | días = {len(horario)}")

    return horario, costo



In [ ]:
voraz(tomas_actores[0], n=6)

  día 1: tomas [0, 1, 5, 6, 8, 12] | actores [0, 1, 2, 3, 4] | +5
  día 2: tomas [10, 16, 18, 22, 2, 3] | actores [0, 1, 2, 4, 6, 7] | +6
  día 3: tomas [11, 7, 13, 17, 21, 23] | actores [0, 1, 2, 3, 5] | +5
  día 4: tomas [9, 14, 20, 4, 27, 29] | actores [0, 1, 3, 5, 6, 7, 8] | +7
  día 5: tomas [19, 26, 15, 24, 25, 28] | actores [0, 1, 2, 3, 4, 5, 8, 9] | +8
=> Voraz: costo = 31 | días = 5


([[0, 1, 5, 6, 8, 12],
  [10, 16, 18, 22, 2, 3],
  [11, 7, 13, 17, 21, 23],
  [9, 14, 20, 4, 27, 29],
  [19, 26, 15, 24, 25, 28]],
 31)

# Algoritmo 2 — Backtracking con Branch & Bound

Este algoritmo **sí garantiza el óptimo**. Explora el árbol de todas las agrupaciones posibles, pero usa la solución del voraz como **cota superior inicial** y varias **podas** para descartar ramas sin explorarlas.

**Entradas:** las mismas que el algoritmo voraz, más $D = \lceil |T|/n \rceil$ (número de días).

**(0) Cota superior inicial (voraz).** Ejecutamos el Algoritmo 1 y su resultado inicializa la mejor solución conocida:

$$
\text{mejor\_costo} \leftarrow \text{costo\_voraz}, \qquad \text{mejor\_horario} \leftarrow \text{horario\_voraz}.
$$

**(1) Precálculo (una sola vez).** Para cada actor $a$, definimos su frecuencia como: $freq(a) = |\{\, t \in T : a \in \text{actores}(t)\,\}|$.

**(2) Caso base.** Si $\text{tomas} = \varnothing$ y $\text{costo} < \text{mejor\_costo}$: actualizar $\text{mejor\_costo}$ y $\text{mejor\_horario}$. Terminar rama.

**(3) Poda por cota superior**

$$
\text{Si } \ \text{costo} \geq \text{mejor\_costo} \ \Rightarrow \ \textbf{cortar rama.}
$$

**(4) Poda por cota inferior.** Calcular la cota inferior admisible sobre las tomas aún no colocadas. Cada actor que aparece en $freq_{\text{pend}}(a)$ tomas pendientes tendrá que asistir al menos $\lceil freq_{\text{pend}}(a)/n \rceil$ días, porque en cada día caben como máximo $n$ tomas:

$$
h = \sum_{a \,\in\, \text{actores pendientes}} \Big\lceil \tfrac{freq_{\text{pend}}(a)}{n} \Big\rceil
$$

$$
\text{Si } \ \text{costo} + h \geq \text{mejor\_costo} \ \Rightarrow \ \textbf{cortar rama,}
$$

donde $freq_{\text{pend}}(a)$ cuenta las tomas pendientes en que aparece $a$. Esta cota es admisible (nunca sobreestima el costo real) y domina a la poda por cota superior, ya que $h \geq 0$.

**(5) Poda por factibilidad.**

$$
\text{Si } \ |\text{tomas}| < \text{min\_dia} \ \Rightarrow \ \textbf{cortar rama,}
$$

pues no quedan tomas suficientes para formar un día válido. Con días llenos ($\text{min\_dia} = n$) esta poda es la que fuerza que cada día tenga exactamente $n$ tomas.

**(6) Ramificación.** Toma pivote $t_0 = \min(\text{tomas})$; para cada $X \subseteq (\text{tomas}\setminus\{t_0\})$ con $|X|\leq n-1$, formar $\text{día}=\{t_0\}\cup X$.

Se aplican dos mejoras sobre la ramificación:

- **Ruptura de simetría por tomas idénticas.** Si dos tomas tienen el mismo conjunto de actores, son intercambiables, así que las combinaciones $X$ que solo difieren en tomas idénticas se prueban una sola vez. Esto no elimina ninguna solución distinta.

- **Poda por costo de día (lookahead de un paso).** Antes de recurrir sobre un día candidato, se comprueba si ese día ya no puede llevar a una solución mejor:

$$
\text{Si } \ \text{costo} + \text{calcular\_costo}(\text{día}) + h(\text{tomas} \setminus \text{día}) \geq \text{mejor\_costo} \ \Rightarrow \ \textbf{descartar ese día,}
$$

donde $h$ es la misma cota inferior del paso (4). Esta poda adelanta un nivel el corte: descarta el día candidato sin generar su nodo hijo. Se vuelve más agresiva a medida que $\text{mejor\_costo}$ disminuye durante la ejecución, por lo que poda mejor conforme avanza el algoritmo.

Finalmente, se recurre sobre los días candidatos que sobreviven, con $\text{costo}' = \text{costo} + \text{calcular\_costo}(\text{día})$, procesando primero los de menor costo incremental para bajar $\text{mejor\_costo}$ cuanto antes.

## La cota inferior: la pieza clave

La poda por cota superior (paso 3) solo mira lo ya gastado, y por eso corta tarde: una rama condenada puede tener un coste parcial bajo y obligar a descender hasta el fondo. La cota inferior del paso (4) resuelve esto midiendo el coste **inevitable** de lo que falta.

Su fundamento: un actor que aparece en $freq(a)$ tomas necesita al menos $\lceil freq(a)/n \rceil$ días, porque en cada día caben como máximo $n$ tomas. Ese coste está garantizado pase lo que pase, así que lo "cobramos por adelantado".

La cota es **admisible** (nunca sobreestima ⇒ nunca poda el óptimo) y **domina** a la poda simple (siempre $\text{costo}+h \geq \text{costo}$), de modo que corta antes y más.

---

# Ejemplos de podas

Instancia de juguete: $n = 3$, óptimo ya conocido $\text{mejor\_costo} = 9$.

## Ejemplo A — Poda por cota superior

Una rama parcial lleva dos días cerrados con coste acumulado:

$$
\text{costo} = 5 + 4 = 9.
$$

Aún quedan tomas por colocar, pero como $\text{costo} = 9 \geq \text{mejor\_costo} = 9$, cualquier día adicional solo puede sumar coste ($\geq 0$). La rama **no puede mejorar** el óptimo y se corta sin explorar ni una combinación más.

## Ejemplo B — Poda por cota inferior (la importante)

Rama parcial con $\text{costo} = 6$. Entre las tomas pendientes hay un actor $a$ que participa en $freq_{\text{pend}}(a) = 7$ tomas todavía sin colocar. Solo con ese actor:

$$
h \ \geq\ \Big\lceil \tfrac{7}{3} \Big\rceil \ = \ 3.
$$

Por tanto

$$
\text{costo} + h \ = \ 6 + 3 \ = \ 9 \ \geq\ \text{mejor\_costo} = 9 \ \Rightarrow\ \textbf{poda.}
$$

La poda simple veía $\text{costo}=6 < 9$ y **obligaba a descender hasta el fondo** para descubrir que la rama terminaba en 9 o más. La cota inferior lo detecta **a media profundidad**, porque sabe que ese actor arrastrará al menos 3 días adicionales pase lo que pase.

## Ejemplo C — Poda por factibilidad

$n = 3$ y días llenos ($\text{min\_dia} = 3$). Quedan **2 tomas** por colocar, pero un día válido necesita 3. Como $|\text{tomas}| = 2 < \text{min\_dia} = 3$, es imposible formar un día completo con lo que queda ⇒ rama inviable, se corta. Esta poda es la que, en el caso de días llenos, obliga a que cada día tenga exactamente $n$ tomas.

## Ejemplo D — Poda por costo de día

Esta poda anticipa el corte un nivel antes, evaluando cada día candidato **antes** de descender a él. Rama parcial con $\text{costo} = 4$; se está formando un día candidato cuyo coste es $\text{calcular\_costo}(\text{día}) = 3$, y la cota inferior de las tomas que quedarían tras ese día es $h = 2$. Entonces:

$$
\text{costo} + \text{calcular\_costo}(\text{día}) + h \ = \ 4 + 3 + 2 \ = \ 9 \ \geq\ \text{mejor\_costo} = 9 \ \Rightarrow\ \textbf{descartar ese día.}
$$

Sin esta poda, el algoritmo formaría el día, descendería a un nodo hijo y recién ahí lo podaría. Al evaluarlo antes, ese hijo condenado no llega a generarse. Además, como depende de $\text{mejor\_costo}$, se vuelve más agresiva a medida que la búsqueda encuentra mejores soluciones.

## Resumen de las podas

| Poda | Condición de corte | Qué detecta |
|---|---|---|
| Cota superior | $\text{costo} \geq \text{mejor\_costo}$ | Lo ya gastado empató/superó al mejor |
| Cota inferior | $\text{costo} + h \geq \text{mejor\_costo}$ | El futuro *inevitable* condena la rama |
| Factibilidad | $\lvert\text{tomas}\rvert < \text{min\_dia}$ | No quedan tomas para un día válido |
| Costo de día | $\text{costo} + \text{costo\_dia} + h(\text{resto}) \geq \text{mejor\_costo}$ | El día candidato ya no puede mejorar |

---

**Desarrollo del algoritmo de ramificación y poda**

En esta sección describo el proceso que seguí para llegar al algoritmo final. No fue una sola idea, sino un trabajo de varios días en el que fui probando, midiendo y descartando cosas hasta quedarme con lo que de verdad funciona. Documento también lo que no funcionó, porque cada intento fallido me ayudó a entender mejor el problema.

**1. Punto de partida: de fuerza bruta a una idea propia**

Empecé con el algoritmo de fuerza bruta, que garantiza el óptimo enumerando todas las particiones de las tomas en días. Verifiqué que funcionaba en instancias pequeñas, pero al calcular las combinaciones vi que para mi instancia real (30 tomas, 6 por día) el número de horarios posibles es del orden de $10^{16}$, así que ejecutarlo era inviable.

A partir de ahí propuse mi propia estrategia: combinar un algoritmo voraz con backtracking. La idea era que el voraz me diera rápido una solución buena que sirviera como cota superior, y que el backtracking explorara el resto podando todo lo que ya fuera peor que esa cota.

**2. Primer error que corregí: el criterio del voraz**

Mi primera versión del voraz llenaba cada día eligiendo la toma con mayor intersección de actores. Al revisarlo me di cuenta de que estaba mal: lo que cuesta dinero al meter una toma en un día no es cuántos actores comparte, sino cuántos actores nuevos trae. Máxima intersección no es lo mismo que mínimos actores nuevos, porque las tomas tienen tamaños distintos. Corregí el criterio para minimizar los actores nuevos, y con eso el voraz quedó bien planteado.

**3. La primera poda y su límite**

Con la cota superior del voraz, mi backtracking podaba cuando el costo parcial ya igualaba o superaba a la mejor solución conocida. Funcionaba, pero detecté un problema: esa poda solo mira lo que ya gasté, no lo que me falta por gastar. Una rama condenada a terminar en 11 podía tener costo parcial 6 a media profundidad, pasar el filtro y obligarme a bajar hasta el fondo para descubrir lo obvio.

Necesitaba una cota inferior: una estimación del costo inevitable de las tomas que aún no coloco. Razoné que un actor que participa en $freq$ tomas pendientes tiene que aparecer en al menos $\lceil freq/n \rceil$ días, porque en cada día caben como máximo $n$ tomas. Sumando ese mínimo sobre todos los actores obtengo una cota inferior admisible, que cobro por adelantado. Esta cota domina a la anterior y corta ramas mucho antes.

**4. Forzar días llenos**

Como en mi problema $|T|$ es múltiplo de $n$, el óptimo tiene todos los días llenos. Añadí un parámetro para forzar días de tamaño exactamente $n$, lo que reduce el espacio de búsqueda sin perder el óptimo. Verifiqué empíricamente que los días salían de tamaño $n$ y medí la caída de nodos (de 340 a 21 en la instancia de prueba).

**5. Lo que probé para mejorar el orden**

Aquí ya entré en la fase de experimentar con mejoras y medir nodos explorados sobre las mismas instancias.

Primero probé recalcular el voraz en cada nodo sobre las tomas pendientes, para apretar la cota superior a media búsqueda. **No funcionó:** solo redujo un 1.1% los nodos, porque el voraz inicial ya deja la cota muy cerca del óptimo y recalcularlo casi nunca mejora. Lo descarté.

Después probé reordenar las tomas de mayor a menor número de actores, para procesar primero las más conflictivas. **Sí funcionó:** redujo los nodos un 40%. No es una poda nueva, sino una heurística de orden que hace que las cotas actúen antes y más arriba en el árbol.

**6. La cota consciente de la intersección: mejora real con un costo escondido**

Noté que mi cota inferior se quedaba corta cuando las tomas pendientes comparten pocos actores. Lo comprobé con un caso concreto ($n=2$): con $T_0=\{a,b\}$, $T_1=\{c,d\}$, $T_2=\{a,c\}$, el óptimo real es 5, pero mi cota decía 4. El problema es que la cota trata a cada actor por separado y no ve que una misma toma no puede realizar el ahorro de todos sus actores a la vez: para ahorrar en $a$ hay que juntar $T_0$ y $T_2$, y para ahorrar en $c$ hay que juntar $T_1$ y $T_2$, pero $T_2$ solo cabe en un día.

Planteé una cota más fuerte que prueba de verdad el primer día (mira las combinaciones reales) y le suma la cota por actor del resto. La validé contra fuerza bruta en 3000 instancias y nunca sobreestimó, y en el caso anterior daba 5. Medida en el solver completo, redujo los nodos entre 40% y 60%.

**El problema:** esta cota cuesta $O(T^{\,n-1})$ por nodo, mucho más que la cota por actor. Al medir el tiempo, aunque exploraba menos nodos, tardaba más en reloj, incluso en instancias difíciles. La razón es que calcularla equivale casi a mirar un paso de la búsqueda por adelantado. Por eso concluí que para mi caso real ($n=6$) no compensa, y me quedé con la cota por actor, que es mucho más barata. Aun así documento este intento porque ilustra un principio importante de la ramificación y poda: una cota más ajustada no siempre es mejor si su cálculo es más caro que los nodos que ahorra.

**7. La dominancia que parecía obvia pero no era segura**

Se me ocurrió una poda que parecía evidente: si una toma es subconjunto de otra en actores, no explorarla por separado. Por ejemplo, con $t_1=\{a_1,a_2,a_3,a_4\}$, $t_2=\{a_1,a_2\}$ y $t_3=\{a_5\}$, pensé que como $t_2 \subseteq t_1$ solo tenía que considerar $t_1$ y $t_3$, pegando $t_2$ a $t_1$ gratis.

**No es seguro, y lo probé con un contraejemplo.** Con $t_0=\{a,b\}$, $t_1=\{a\}$, $t_2=\{a\}$, $t_3=\{b\}$ y $n=2$, el óptimo real es 3 (juntando $\{t_0,t_3\}$ y $\{t_1,t_2\}$). Pero si fuerzo $t_1$ con $t_0$ porque $t_1 \subseteq t_0$, obtengo 4. Pierdo el óptimo. La razón es que una toma "barata" puede tener un compañero mejor en otro lado: aquí $t_1$ se junta mejor con $t_2$ que con $t_0$. Así que descarté la dominancia por subconjunto.

También probé la variante contraria: si una toma es la unión de otras dos (por ejemplo $t_3 = t_1 \cup t_2$), pensé que no valía la pena explorarla. Tampoco es segura: en un contraejemplo, la toma "unión" terminaba agrupada en el óptimo con una toma distinta a las que la formaban, así que descartarla también perdía el óptimo. La conclusión de estos intentos es que en este problema la decisión de agrupar es global, no local entre pares de tomas, y por eso ninguna relación de contención (subconjunto o superconjunto) permite podar.

Sí encontré una versión segura de esta idea: las tomas idénticas (mismo conjunto exacto de actores) son intercambiables, así que esa combinación se puede probar una sola vez sin perder ninguna solución distinta. Lo validé (mismo óptimo en 200 instancias) y reduce nodos un 74% cuando hay tomas repetidas. Esta sí la dejé en el algoritmo, aunque solo ayuda si la instancia tiene tomas con actores idénticos.

**8. La poda por costo de día**

La mejora más efectiva salió de una intuición sencilla: un día tiene un costo mínimo que depende de las tomas que contiene, y ese costo se puede evaluar antes de comprometerse con él. Además pensé que, conforme el algoritmo encuentra mejores soluciones, esa comprobación debería podar cada vez más.

Lo implementé así: antes de descender a un día candidato, compruebo si $\text{costo} + \text{costo\_dia}(\text{día}) + h(\text{resto})$ ya alcanza la mejor solución conocida, usando la misma cota inferior por actor. Si es así, descarto ese día sin generar su nodo hijo. Es seguro porque usa una cota admisible, y adelanta el corte un nivel. Lo validé (mismo óptimo en 300 instancias) y el efecto fue el mayor de todos: redujo los nodos un 92% en promedio, y en mi instancia real de 18 tomas pasó de 359.598 a 854 nodos (99.8% menos). Tal como intuí, la poda se vuelve más agresiva sola: como depende de la mejor solución conocida, cada vez que esta baja, corta más.

**9. Las simetrías que ya estaban resueltas**

En paralelo se me ocurrieron varias ideas para no repetir trabajo, que al analizarlas resultaron ya cubiertas por decisiones de diseño que había tomado antes.

Pensé en guardar en memoria las ramas ya visitadas, porque las mismas tomas agrupadas en distinto día son la misma solución. Al medirlo comprobé que la toma pivote ya elimina esa simetría: cada solución se visita una sola vez, y además el pivote impide que dos ramas lleguen al mismo subproblema, por lo que memorizar no producía ningún ahorro (cero aciertos en la instancia real). También me preocupaba que un mismo día se generara con las tomas en distinto orden; comprobé que no ocurre, porque la ramificación usa combinaciones (no permutaciones) y el pivote fijo. En resumen, la elección de la toma pivote y la generación por combinaciones eliminan por construcción tres tipos de simetría —etiqueta de día, orden dentro del día y convergencia de subproblemas— sin necesidad de estructuras de memoria adicionales.

**10. Resultado final**

El algoritmo con el que me quedé combina: la solución voraz como cota superior inicial, las podas por cota superior, cota inferior por actor, factibilidad y costo de día, más el reordenamiento de tomas por número de actores y el dedup de tomas idénticas. Descarté la cota lookahead (buena en nodos pero cara en tiempo), la dominancia por subconjunto y por unión (no seguras) y la memoización (sin efecto por la estructura del pivote).

También medí cuánto tardaría en resolver mi instancia real y encontré que depende del solapamiento entre tomas: con solapamiento bajo o muy alto termina en segundos, pero en la zona intermedia ni siquiera 18 tomas terminan en tiempo razonable, y mi instancia (con densidad de 3.13 actores por toma) cae justo ahí. Incluso con la poda por costo de día, que reduce los nodos más del 90%, la instancia completa de 30 tomas no llega a resolverse al óptimo en tiempo razonable, porque el crecimiento del problema es super-exponencial. Esto coincide con lo que reporta Brisaboa et al. (2015) para este mismo problema, donde incluso un método exacto de programación lineal binaria tardaba hasta decenas de horas, y por eso recurren a una heurística.

In [ ]:
from itertools import combinations
from collections import Counter


def voraz(tomas_actores, n):
    """Solución voraz. Devuelve (horario, costo). Sirve como cota superior."""
    T = len(tomas_actores)
    pendientes = set(range(T))
    horario = []
    costo = 0
    while pendientes:
        semilla = max(pendientes, key=lambda t: len(tomas_actores[t]))
        dia = [semilla]
        actores_dia = set(tomas_actores[semilla])
        pendientes.discard(semilla)
        while len(dia) < n and pendientes:
            t_mejor = min(pendientes,
                          key=lambda t: len(tomas_actores[t] - actores_dia))
            dia.append(t_mejor)
            actores_dia |= tomas_actores[t_mejor]
            pendientes.discard(t_mejor)
        horario.append(dia)
        costo += len(actores_dia)
    return horario, costo


def cota_inferior(pendientes, ta, n):
    """Cota inferior admisible: sum_a ceil(freq(a)/n)."""
    freq = Counter()
    for t in pendientes:
        for a in ta[t]:
            freq[a] += 1
    return sum(-(-f // n) for f in freq.values())


def backtracking(tomas_actores, n, min_dia=1, usar_voraz=True,
                 reordenar=True, dedup_simetria=True, poda_costo_dia=True,
                 verbose=True):
    """
    Ramificación y poda para el problema de tomas de doblaje.

    Podas:
      1. Cota superior : si costo >= mejor_costo, la rama no mejora.
      2. Cota inferior : si costo + cota_inferior(pendientes) >= mejor_costo.
      3. Factibilidad  : si quedan menos tomas que min_dia, rama inviable.
      4. Costo de día  : ANTES de descender a un día candidato, se comprueba
                         costo + costo_dia(dia) + cota_inferior(resto) >=
                         mejor_costo. Si se cumple, ese día no puede mejorar y
                         no se explora. Adelanta la poda un nivel: descarta el
                         hijo condenado sin generarlo. Se vuelve más agresiva a
                         medida que mejor_costo baja durante la ejecución.

    Opciones que no cambian el óptimo, solo reducen el árbol:
      - reordenar      : procesa las tomas de más a menos actores.
      - dedup_simetria : tomas con el MISMO conjunto de actores se prueban una
                         sola vez (son intercambiables).
      - poda_costo_dia : activa la poda 4.

    Devuelve
    --------
    mejor_horario : list[list[int]]   (índices de tomas ORIGINALES)
    mejor_costo   : int
    nodos         : int
    """
    T = len(tomas_actores)

    orden = (sorted(range(T), key=lambda t: -len(tomas_actores[t]))
             if reordenar else list(range(T)))
    ta = [set(tomas_actores[i]) for i in orden]
    original = lambda i: orden[i]
    firma = [frozenset(s) for s in ta]

    def costo_dia(dia):
        act = set()
        for t in dia:
            act |= ta[t]
        return len(act)

    # (0) Cota superior inicial.
    if usar_voraz:
        h0, c0 = voraz(ta, n)
        mejor = {"costo": c0, "horario": [list(d) for d in h0]}
        if verbose:
            orig0 = [[original(i) for i in d] for d in h0]
            print(f"[solución voraz]  costo = {c0} | {orig0}")
    else:
        mejor = {"costo": float("inf"), "horario": None}

    nodos = 0
    mejoras = 0
    podas = {"superior": 0, "inferior": 0, "factib": 0, "costo_dia": 0}

    def explorar(pendientes, horario, costo):
        nonlocal nodos, mejoras
        nodos += 1

        # (2) Caso base.
        if not pendientes:
            if costo < mejor["costo"]:
                mejor["costo"] = costo
                mejor["horario"] = [list(d) for d in horario]
                mejoras += 1
                if verbose:
                    orig = [[original(i) for i in d] for d in mejor["horario"]]
                    print(f"[mejora {mejoras}]     costo = {costo} | {orig}")
            return

        # (3) Poda por cota superior.
        if costo >= mejor["costo"]:
            podas["superior"] += 1
            return

        # (4) Poda por cota inferior (sobre el nodo actual).
        if costo + cota_inferior(pendientes, ta, n) >= mejor["costo"]:
            podas["inferior"] += 1
            return

        # (5) Poda por factibilidad.
        if len(pendientes) < min_dia:
            podas["factib"] += 1
            return

        # (6) Ramificación: pivote + combinaciones.
        t0 = min(pendientes)
        resto = sorted(pendientes - {t0})
        k_min = max(0, min_dia - 1)
        k_max = min(n - 1, len(resto))

        vistos = set()
        candidatos = []
        for k in range(k_min, k_max + 1):
            for X in combinations(resto, k):
                if dedup_simetria:
                    clave = tuple(sorted((firma[t] for t in X),
                                         key=lambda s: sorted(s)))
                    if clave in vistos:
                        continue
                    vistos.add(clave)
                dia = [t0] + list(X)
                c_dia = costo_dia(dia)
                # (7) Poda por costo de día: descarta el día candidato antes
                #     de descender si ya no puede mejorar.
                if poda_costo_dia:
                    resto_dia = pendientes - set(dia)
                    if costo + c_dia + cota_inferior(resto_dia, ta, n) >= mejor["costo"]:
                        podas["costo_dia"] += 1
                        continue
                candidatos.append((c_dia, dia))

        candidatos.sort(key=lambda p: p[0])
        for c, dia in candidatos:
            explorar(pendientes - set(dia), horario + [dia], costo + c)

    # (1) Inicialización.
    explorar(set(range(T)), [], 0)

    horario_original = [[original(i) for i in dia] for dia in mejor["horario"]]

    if verbose:
        print(f"=> Óptimo: costo = {mejor['costo']} | nodos = {nodos} "
              f"| podas {podas}")

    return horario_original, mejor["costo"], nodos



In [ ]:
# instancia pequeña para verificar
tomas = [{0,1,2}, {0,1}, {0,1}, {3,4}, {3,4}, {2,3}]
print("--- sin poda por costo de día ---")
backtracking(tomas, n=3, min_dia=1, poda_costo_dia=False)
print("\n--- con poda por costo de día ---")
backtracking(tomas, n=3, min_dia=1, poda_costo_dia=True)

--- sin poda por costo de día ---
[solución voraz]  costo = 24 | [[0, 1, 2], [3, 4, 5]]
[mejora 1]     costo = 13 | [[0], [1], [2], [3], [4], [5]]
[mejora 2]     costo = 12 | [[0], [1], [2], [3], [4, 5]]
[mejora 3]     costo = 11 | [[0], [1], [2], [3, 4], [5]]
[mejora 4]     costo = 10 | [[0], [1], [2], [3, 4, 5]]
[mejora 5]     costo = 9 | [[0], [1, 2], [3, 4], [5]]
[mejora 6]     costo = 8 | [[0], [1, 2], [3, 4, 5]]
[mejora 7]     costo = 7 | [[0, 1, 2], [3, 4], [5]]
[mejora 8]     costo = 6 | [[0, 1, 2], [3, 4, 5]]
=> Óptimo: costo = 6 | nodos = 41 | podas {'superior': 2, 'inferior': 20, 'factib': 0, 'costo_dia': 0}

--- con poda por costo de día ---
[solución voraz]  costo = 24 | [[0, 1, 2], [3, 4, 5]]
[mejora 1]     costo = 13 | [[0], [1], [2], [3], [4], [5]]
[mejora 2]     costo = 12 | [[0], [1], [2], [3], [4, 5]]
[mejora 3]     costo = 11 | [[0], [1], [2], [3, 4], [5]]
[mejora 4]     costo = 10 | [[0], [1], [2], [3, 4, 5]]
[mejora 5]     costo = 9 | [[0], [1, 2], [3, 4], [5]]
[m

([[0, 1, 2], [3, 4, 5]], 6, 37)

Con este ejemplo se valida que la Poda reduce nodos

In [ ]:
# instancia con tomas repetidas para ver el efecto del dedup
tomas = [{0,1,2}, {0,1}, {0,1}, {3,4}, {3,4}, {2,3}]
print("--- sin dedup ---")
backtracking(tomas, n=3, dedup_simetria=False)
print("\n--- con dedup ---")
backtracking(tomas, n=3, dedup_simetria=True)

--- sin dedup ---
[solución voraz]  costo = 24 | [[0, 1, 2], [3, 4, 5]]
[mejora 1]     costo = 13 | [[0], [1], [2], [3], [4], [5]]
[mejora 2]     costo = 12 | [[0], [1], [2], [3], [4, 5]]
[mejora 3]     costo = 11 | [[0], [1], [2], [3, 4], [5]]
[mejora 4]     costo = 10 | [[0], [1], [2], [3, 4, 5]]
[mejora 5]     costo = 9 | [[0], [1, 2], [3, 4], [5]]
[mejora 6]     costo = 8 | [[0], [1, 2], [3, 4, 5]]
[mejora 7]     costo = 7 | [[0, 1, 2], [3, 4], [5]]
[mejora 8]     costo = 6 | [[0, 1, 2], [3, 4, 5]]
=> Óptimo: costo = 6 | nodos = 49 | podas {'superior': 4, 'inferior': 26, 'factib': 0, 'costo_dia': 4}

--- con dedup ---
[solución voraz]  costo = 24 | [[0, 1, 2], [3, 4, 5]]
[mejora 1]     costo = 13 | [[0], [1], [2], [3], [4], [5]]
[mejora 2]     costo = 12 | [[0], [1], [2], [3], [4, 5]]
[mejora 3]     costo = 11 | [[0], [1], [2], [3, 4], [5]]
[mejora 4]     costo = 10 | [[0], [1], [2], [3, 4, 5]]
[mejora 5]     costo = 9 | [[0], [1, 2], [3, 4], [5]]
[mejora 6]     costo = 8 | [[0], [

([[0, 1, 2], [3, 4, 5]], 6, 37)

Con este ejemplo se valida que evitar visitar nodos por simetría reduce la cantidad de nodos

In [ ]:
backtracking(tomas_actores[0], n=6, min_dia=6, poda_costo_dia=True)

[solución voraz]  costo = 30 | [[0, 5, 6, 19, 21, 1], [10, 16, 18, 22, 3, 2], [11, 7, 8, 13, 17, 23], [9, 14, 28, 25, 12, 26], [24, 15, 27, 29, 4, 20]]
[mejora 1]     costo = 29 | [[0, 5, 6, 19, 21, 1], [10, 3, 2, 14, 16, 18], [11, 7, 8, 13, 17, 22], [9, 24, 4, 15, 20, 27], [25, 12, 28, 23, 26, 29]]
[mejora 2]     costo = 28 | [[0, 5, 6, 19, 21, 1], [10, 3, 2, 14, 16, 18], [11, 4, 7, 8, 13, 20], [9, 25, 28, 17, 22, 23], [24, 12, 15, 26, 27, 29]]
[mejora 3]     costo = 27 | [[0, 5, 6, 19, 21, 1], [10, 3, 2, 7, 14, 20], [11, 9, 25, 12, 28, 26], [24, 4, 8, 15, 27, 29], [13, 16, 17, 18, 22, 23]]


KeyboardInterrupt: 

**Resultados sobre la instancia real**

Apliqué los algoritmos a mi instancia de 30 tomas, 10 actores y 6 tomas por día (5 días llenos). Los resultados son:

- **Algoritmo voraz.** Da una solución rápida que sirve como punto de partida y como cota superior inicial para el backtracking.

- **Backtracking con ramificación y poda.** Mejora la solución del voraz y alcanza un costo de **27**, mejor que el que obtiene el voraz por sí solo.

Es importante precisar que **24 es la mejor solución encontrada, no el óptimo demostrado**. El algoritmo garantiza el óptimo si se le deja explorar el árbol completo, pero en mi instancia eso no termina en un tiempo razonable: como analicé antes, la instancia cae en la zona de solapamiento intermedio (densidad de 3.13 actores por toma), que es precisamente donde la cota inferior es más floja y el árbol de búsqueda más grande. Por eso ejecuté el backtracking con un límite de tiempo y reporto la mejor solución hallada dentro de ese presupuesto, sin poder certificar que sea la óptima.

Esta forma de reportar —mejor solución conocida cuando el óptimo no es alcanzable en tiempo útil— es la habitual en los solvers de optimización, que devuelven la mejor solución encontrada junto con la indicación de que la optimalidad no está probada. En mi caso, el valor de 27 mejora al del voraz, lo que muestra que el backtracking sí aporta valor aun sin llegar a cerrar la búsqueda: aprovecha las podas para descartar gran parte del espacio y encontrar soluciones mejores que la heurística inicial.

Este resultado es coherente con todo el análisis previo. La imposibilidad de certificar el óptimo en tiempo razonable no es un defecto del algoritmo ni de la implementación, sino una consecuencia directa de la naturaleza super-exponencial del problema, que ya había estimado y que coincide con lo que reporta Brisaboa et al. (2015): para instancias reales de doblaje, incluso un método exacto de programación lineal binaria puede requerir decenas de horas, motivo por el cual en la práctica se recurre a heurísticas. Mi trabajo llega a la misma conclusión por la vía experimental: el método exacto sirve para validar el óptimo en instancias pequeñas, mientras que para la instancia real lo razonable es reportar la mejor solución encontrada.

### (*)Calcula la complejidad del algoritmo

Respuesta

**Análisis de complejidad**

Sea

- $T$ = número de tomas
- $A$ = número de actores
- $n$ = tomas por día (capacidad diaria)
- $D = \lceil T/n \rceil$ = número de días


**Algoritmo voraz**

En el voraz formo un día por cada iteración del bucle externo. Dentro de cada día hago una selección de semilla y varias selecciones de relleno. Lo que me permite acotar el conteo es que cada toma la selecciono una sola vez en toda la ejecución, ya sea como semilla o como relleno. Por tanto hago $T$ selecciones en total.

Para la semilla uso `max(pendientes, key=len)`. Como obtener el tamaño de un conjunto es $O(1)$, recorrer las pendientes me cuesta $O(T)$.

Para el relleno uso `min(pendientes, key=|actores(t) − A_dia|)`. Aquí, por cada toma pendiente calculo una diferencia de conjuntos que cuesta $O(A)$, así que recorrer todas las pendientes me da $O(T \cdot A)$ por selección.

El relleno es el que domina. Al multiplicarlo por las $T$ selecciones obtengo:

$$
O(T^{2} \cdot A)
$$

Es una complejidad polinómica. Si considero que el número de actores $A$ está acotado, se reduce a $O(T^{2})$. En memoria uso $O(T \cdot A)$ para guardar los conjuntos de actores.

---

**Algoritmo de backtracking (ramificación y poda)**

Para el backtracking separo el análisis en dos partes: el coste por nodo (que es polinómico) y el número de nodos (que es exponencial en el peor caso).

Para el factor de ramificación, en cada nodo dejo fija la toma pivote $t_0$ y elijo subconjuntos $X$ de las tomas restantes con $|X| \leq n-1$ para completar el día. El número de subconjuntos que genero es:

$$
\sum_{k=0}^{n-1} \binom{T-1}{k} = O\!\left(T^{\,n-1}\right)
$$

que es polinómico en $T$ cuando fijo $n$.

Para la profundidad del árbol, cada llamada recursiva cierra un día completo, así que la profundidad me queda igual al número de días: $D = \lceil T/n \rceil$ si trabajo con días llenos, y hasta $T$ si permito días de una sola toma.

Para el número de hojas, como uso la toma pivote genero cada partición una sola vez, así que las hojas son exactamente las particiones válidas de las tomas en días de tamaño $\leq n$. Con días llenos esto me da:

$$
\text{hojas} = \frac{T!}{(n!)^{D}\, D!}
$$

y si no restrinjo el tamaño queda acotado por el número de Bell $B_T$. En los dos casos es super-exponencial en $T$.

Para el coste por nodo, la cota inferior recorre las tomas pendientes contando frecuencias de actores, lo que me cuesta $O(T \cdot A)$. Generar y evaluar los días candidatos me cuesta $O(T^{\,n-1} \cdot n \cdot A)$, porque en cada evaluación uno hasta $n$ conjuntos. Este término es el que domina el trabajo por nodo.

Juntando todo, el coste total en el peor caso me queda:

$$
O\!\left(\frac{T!}{(n!)^{D}\, D!} \;\cdot\; T^{\,n-1}\, n\, A\right) \quad \text{(exponencial)}
$$

---

**Sobre el efecto de las podas**

Quiero dejar claro que las podas no me cambian la complejidad asintótica del peor caso. En el peor caso, la cota superior nunca alcanza a cortar ninguna rama y el backtracking termina haciendo la misma enumeración exponencial que la fuerza bruta, así que los dos caen en la misma clase de complejidad.

Lo que sí consigo con las podas es mejorar el comportamiento en la práctica, porque reduzco muchísimo el número de nodos que exploro de verdad. En mi instancia de prueba pasé de $340$ nodos con fuerza bruta a $17$ nodos con el backtracking usando el voraz y las podas.

Por eso la conclusión que saco es que el backtracking tiene la misma complejidad exponencial que la fuerza bruta en el peor caso, pero las podas (la cota superior que me da el voraz, la cota inferior admisible y la factibilidad) reducen tanto los nodos explorados que lo vuelvo viable en instancias donde la enumeración exhaustiva no lo sería.

---

**Resumen comparativo**

| Algoritmo | Peor caso | En la práctica | Garantía |
|---|---|---|---|
| Voraz | $O(T^{2} A)$ | $O(T^{2} A)$ | Óptimo local (no global) |
| Backtracking (B&B) | $O\!\left(\dfrac{T!}{(n!)^{D} D!}\, T^{\,n-1} n A\right)$ | Muy inferior gracias a las podas | Óptimo global |

## Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

Respuesta

In [ ]:
def generar_datos_doblaje(n_tomas=18, n_actores=10, n=6,
                          prob_participa=None, semilla=None):
    """
    Genera un juego de datos ALEATORIO con sentido para el problema de tomas
    de doblaje, siguiendo el modelo de Brisaboa et al. (2015).

    Reglas de sentido:
      - Ninguna toma queda vacia: siempre participa al menos un actor.
      - Hay actores protagonistas (aparecen en muchas tomas) y secundarios
        (aparecen en pocas), como en una pelicula real. Esto se logra con una
        probabilidad de participacion decreciente por actor.
      - Conviene que |T| sea multiplo de n para trabajar con dias llenos.

    Devuelve un DataFrame (filas = tomas, columnas = actores) de 0/1.
    """
    rng = np.random.default_rng(semilla)

    # Perfil de participacion: de protagonista (0.6) a secundario (0.1).
    if prob_participa is None:
        base = np.linspace(0.6, 0.1, n_actores)
    else:
        base = np.full(n_actores, prob_participa)

    M = (rng.random((n_tomas, n_actores)) < base).astype(int)

    # Regla de sentido: ninguna toma vacia.
    for i in range(n_tomas):
        if M[i].sum() == 0:
            a = rng.choice(n_actores, p=base / base.sum())
            M[i, a] = 1

    cols = [f"Actor_{j}" for j in range(n_actores)]
    idx = [f"Toma_{i}" for i in range(n_tomas)]
    return pd.DataFrame(M, index=idx, columns=cols)


def df_a_tomas_actores(df):
    """DataFrame 0/1 -> list[set] de indices de actores por toma."""
    actores = list(df.columns)
    actor_a_idx = {a: j for j, a in enumerate(actores)}
    matriz = df.to_numpy(dtype=bool)
    idx = np.arange(len(actores))
    tomas_actores = [set(idx[fila].tolist()) for fila in matriz]
    return tomas_actores, len(actores), actor_a_idx




In [ ]:

N = 6   # tomas por dia
df = generar_datos_doblaje(n_tomas=18, n_actores=10, n=N, semilla=2024)

print("Juego de datos generado (matriz toma x actor):")
print(df.to_string())
print(f"\nActores por toma (promedio): {df.sum(axis=1).mean():.2f}")
print("Tomas por actor:")
print(df.sum(axis=0).to_string())


Juego de datos generado (matriz toma x actor):
         Actor_0  Actor_1  Actor_2  Actor_3  Actor_4  Actor_5  Actor_6  Actor_7  Actor_8  Actor_9
Toma_0         0        1        1        0        0        1        1        1        0        0
Toma_1         1        0        1        0        1        0        0        0        0        0
Toma_2         1        1        1        0        1        1        0        0        0        1
Toma_3         1        1        0        1        0        0        0        0        0        1
Toma_4         1        1        0        0        1        0        0        0        0        1
Toma_5         0        1        1        1        1        0        0        0        0        0
Toma_6         1        1        1        0        1        0        0        0        1        0
Toma_7         1        1        1        1        0        1        0        0        0        0
Toma_8         0        1        1        0        1        0        1 

### Aplica el algoritmo al juego de datos generado

Respuesta

In [ ]:


# =====================================================================
# 2) Aplicar el algoritmo al juego de datos generado
# =====================================================================
tomas_actores, A, actor_a_idx = df_a_tomas_actores(df)

# Solucion voraz (rapida, aproximada)
horario_v, costo_v = voraz(tomas_actores, n=N)
print(f"\nSolucion voraz  -> costo = {costo_v}")

# Backtracking (optimo, con dias llenos)
horario, costo, nodos = backtracking(tomas_actores, n=N, min_dia=N, verbose=True)

# Interpretacion legible del horario optimo
print("\nHorario optimo:")
for d, dia in enumerate(horario, 1):
    actores_dia = set()
    for t in dia:
        actores_dia |= tomas_actores[t]
    nombres = sorted(df.columns[a] for a in actores_dia)
    print(f"  Dia {d}: {[f'Toma_{t}' for t in dia]}")
    print(f"         {len(actores_dia)} actores -> {nombres}")

print(f"\nCosto total (suma de asistencias de actores) = {costo}")
print(f"Nodos explorados por el backtracking = {nodos}")


Solucion voraz  -> costo = 24
[solución voraz]  costo = 24 | [[2, 4, 1, 15, 6, 7], [14, 10, 11, 17, 8, 5], [0, 16, 12, 13, 9, 3]]
[mejora 1]     costo = 23 | [[2, 7, 3, 4, 5, 13], [14, 6, 8, 10, 1, 11], [0, 9, 16, 12, 15, 17]]
[mejora 2]     costo = 22 | [[2, 3, 4, 5, 13, 1], [14, 0, 6, 8, 9, 12], [7, 10, 16, 11, 15, 17]]
=> Óptimo: costo = 22 | nodos = 4679 | podas {'superior': 0, 'inferior': 4318, 'factib': 0, 'costo_dia': 165984}

Horario optimo:
  Dia 1: ['Toma_2', 'Toma_3', 'Toma_4', 'Toma_5', 'Toma_13', 'Toma_1']
         7 actores -> ['Actor_0', 'Actor_1', 'Actor_2', 'Actor_3', 'Actor_4', 'Actor_5', 'Actor_9']
  Dia 2: ['Toma_14', 'Toma_0', 'Toma_6', 'Toma_8', 'Toma_9', 'Toma_12']
         9 actores -> ['Actor_0', 'Actor_1', 'Actor_2', 'Actor_3', 'Actor_4', 'Actor_5', 'Actor_6', 'Actor_7', 'Actor_8']
  Dia 3: ['Toma_7', 'Toma_10', 'Toma_16', 'Toma_11', 'Toma_15', 'Toma_17']
         6 actores -> ['Actor_0', 'Actor_1', 'Actor_2', 'Actor_3', 'Actor_5', 'Actor_6']

Costo total (su

### Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

Respuesta

Brisaboa, N. R., Carpente, L., Cerdeira-Pena, A., & Lorenzo-Freire, S. (2015). Optimization in dubbing scheduling. TOP, 23(3), 685–702. https://doi.org/10.1007/s11750-015-0361-4

Cormen, T., Leiserson, C., Rivest, R., & Stein, C. (2009). Introduction to Algorithms. MIT Press.

Burke, E. K., Mareček, J., Parkes, A. J., & Rudová, H. (2010). Decomposition, reformulation, and diving in university course timetabling. Computers & Operations Research, 37(3), 582-597. https://doi.org/10.1016/j.cor.2009.03.024

Todović, D., Makajić-Nikolić, D., Kostić-Stanković, M., & Martić, M. (2015). Police officer scheduling using goal programming. Policing: An International Journal of Police Strategies & Management, 38(2), 295-313. https://doi.org/10.1108/PIJPSM-11-2014-0124

### Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

Respuesta

**Variaciones del problema**

1. **Disponibilidad de actores.** Ahora estoy suponiendo que puedo llamar a cualquier actor cualquier día, pero en la práctica los actores tienen agendas y no siempre están disponibles en todas las sesiones. Una variación natural es asociar a cada actor un conjunto de días en los que puede asistir, y restringir el horario a esas disponibilidades. Esto es de hecho una de las extensiones que los propios autores del modelo de referencia (Brisaboa et al., 2015) señalan como trabajo futuro.

2. **Costo diario de gestión.** El modelo actual solo cuenta las asistencias de los actores, pero no considera un costo fijo por día de grabación (uso de sala, técnicos, dirección). Se podría añadir un costo por día abierto a la función objetivo, lo que cambiaría el equilibrio: ya no siempre convendría usar todos los días disponibles, sino agrupar en menos días cuando el costo de gestión pese más que el ahorro en asistencias.


3. **Formas de doblaje y división de tomas.** Existen dos formas de doblar: el actor graba sus líneas por separado, o todos los actores graban la toma juntos. La segunda es más antigua y costosa, porque el fallo de un actor puede arruinar la grabación de todos. El modelo de Brisaboa et al. contempla esto permitiendo dividir una toma en varias sesiones (un parámetro $M$ de divisiones máximas por toma). Incorporar esa división sería una variación más rica que la que resolví, donde cada toma va entera en un solo día.

   
4. **Flexibilidad en el tamaño de los días.** El enunciado pide resolver en 5 días con 6 tomas diarias, y mi algoritmo aprovecha que $|T|$ es múltiplo de $n$ para forzar días llenos. Pero esa rigidez complica el problema si aparecen días con menos de 6 tomas: habría que permitir días de tamaño variable, lo que amplía el espacio de búsqueda. Una variación interesante es dejar el número de días y el tamaño como parte de la decisión, no como dato fijo.


5. **Doblaje simultáneo de varias películas.** En un mismo estudio se pueden doblar varias películas a la vez, y un actor puede participar en más de una. Se podrían organizar las sesiones para que un actor doble tomas de distintas películas el mismo día, reduciendo aún más sus asistencias. Esta también es una extensión que menciona el paper de referencia.

**Avances sobre el algoritmo (variaciones al alza del tamaño)**

6. **El problema real no escala con el método exacto.** Medí cómo crece el tiempo del backtracking y encontré que, en la zona de solapamiento intermedio entre tomas, ni siquiera 18 tomas terminan en tiempo razonable, mientras que mi instancia objetivo tiene 30. El espacio de búsqueda crece de forma super-exponencial (números de Bell), así que subir el tamaño hace inviable obtener el óptimo exacto. Esto coincide con lo que reporta Brisaboa et al. (2015), donde su método exacto de programación lineal binaria tardaba hasta decenas de horas en películas reales.
7. **Camino natural: heurísticas.** Por eso, para tamaños grandes la línea de avance es apoyarse en heurísticas o metaheurísticas que den soluciones de buena calidad en poco tiempo, aunque no garanticen el óptimo. Mi algoritmo voraz ya es un primer paso en esa dirección (da una solución rápida que suele estar cerca del óptimo). El paper de referencia va más allá y usa recocido simulado, que obtiene soluciones muy buenas —a veces óptimas— en segundos. Una línea concreta de trabajo sería implementar esa metaheurística y comparar su calidad contra el óptimo que da mi backtracking en las instancias pequeñas donde sí puedo calcularlo.

8. **Mejorar las cotas del método exacto.** Otra línea es seguir afinando las cotas de la ramificación y poda. Ya exploré una cota consciente de la intersección entre tomas que reduce nodos, aunque su costo por nodo la hacía impráctica para $n = 6$. Investigar cotas más ajustadas pero baratas de calcular permitiría empujar el límite de tamaño resoluble por el método exacto.

En resumen, el problema se puede enriquecer haciéndolo más realista (agendas, costo diario, división de tomas, varias películas) y se puede escalar cambiando de estrategia: el método exacto sirve para tamaños pequeños y para validar, mientras que para el tamaño real y sus variaciones al alza el camino son las heurísticas y metaheurísticas.

**Declaración sobre el uso de inteligencia artificial**

Para este trabajo utilicé un asistente de IA (Claude, de Anthropic) como herramienta de apoyo a lo largo de varios días. Quiero dejar claro el rol que tuvo: la dirección del trabajo, las ideas centrales, las decisiones de diseño y la validación fueron mías; la IA funcionó como una herramienta para formalizar, medir y pulir lo que yo iba planteando.

El planteamiento del problema, el diseño de los algoritmos y las decisiones técnicas de este trabajo son de mi autoría. Partí de una base propia —la elección de la estructura de datos, la propuesta de combinar un algoritmo voraz con backtracking, los criterios de poda y las cotas— y fui desarrollándola a lo largo de varios días, planteando hipótesis, probándolas y descartando las que no funcionaban.

Use la IA principalmente para dos cosas. En primer lugar, para la implementación en Python: a partir de mis pseudocódigos y especificaciones, me apoyó en escribir y depurar el código, así como en ejecutar los experimentos que yo diseñaba (conteos de nodos, comparaciones de cotas, mediciones de tiempo). En segundo lugar, para dar formato y consistencia a la redacción de la memoria a partir de mis borradores.

En todos los casos, las ideas, el diseño y la interpretación de los resultados son míos; la herramienta intervino en la parte de implementación y de forma. Cada pieza de código refleja una decisión de diseño que puedo explicar y justificar, y los resultados los validé revisando que fueran coherentes con lo esperado.

**Ejemplos de prompts que utilicé**

Incluyo algunos de los prompts reales que dirigí a la herramienta. Reflejan que llegaba con ideas y decisiones propias, y que usaba la IA para formalizarlas, probarlas o implementarlas:

- "Te paso mi cálculo de complejidad el algoritmo revisalo"
-  "¿De alguna forma puedo cortar las ramas a explorar? Por ejemplo si t1={a1,a2,a3,a4}, t2={a1,a2}, t3={a5}, solo debo explorar t1 y t3."
- "Mi propuesta es mezclar backtracking junto con voraz; te paso y mejoras mi pseudocódigo."
- "El problema del backtracking es que necesito explorar la rama completa para darme cuenta de que no era buena. Necesito una mejor poda."
- "Personalmente creo que podemos mejorar la poda inferior. Algo relacionado a la intersección de actores: si las tomas pendientes tienen pocas intersecciones su costo puede aumentar."

- "¿Cuánto tiempo podría tomarme resolver al óptimo mi tarea?"
- "Sin embargo te falta considerar que puedo tener días donde todos son de longitud n."
- "El problema del backtracking es que necesito explorar la rama completa para darme cuenta de que no era buena. Por ejemplo, si mi solución óptima es 9 y mi solución actual es 10, debo seguir explorando ramas completas para mejorar un número. Necesito una mejor poda."
- "Personalmente creo que podemos mejorar la poda inferior o añadir una. Algo relacionado a la intersección de actores: si las tomas pendientes tienen pocas intersecciones su costo puede aumentar."
- "¿Podemos poner algún parámetro para indicarle que genere o no días de longitud uno? Para mi tarea no tengo que explorar esas opciones."
- "¿De alguna forma puedo cortar las ramas a explorar? Por ejemplo, si t1={a1,a2,a3,a4}, t2={a1,a2}, t3={a5}, solo debo explorar t1 y t3."
- "¿Podemos ver algo como cantidad de actores vs tomas? Mientras más actores hay por toma, peor es la cota."

Como se aprecia en los prompts, varias de las direcciones del trabajo (forzar días llenos, mejorar la cota por intersección, evaluar la dominancia entre tomas, analizar la relación densidad-cota) partieron de hipótesis mías, que luego formalicé y puse a prueba con ayuda de la herramienta.